# 01 — Labels, performance window, splits

The highest-leverage decisions in the project, all made before a model exists.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.config import load_config

cfg = load_config()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

## 1. Three-way label mapping

`loan_status` has nine values in **three** groups. A loan still `Current` has not had the chance to default — it is unresolved, not good.

Verified on the full file: **40.36% of rows are censored**, 878,317 of them `Current`. Mapping those to the good class would poison roughly four rows in ten.

In [ ]:
from src.data_load import load_accepted
from src.labeling import assign_target

acc = load_accepted(cfg['paths']['raw_accepted'],
                    usecols=['issue_d','term','loan_status'])
print(acc['loan_status'].value_counts().to_string())
t = assign_target(acc['loan_status'], cfg['labels'])
print(f'\nbad {int((t==1).sum()):,} | good {int((t==0).sum()):,} | censored {int(t.isna().sum()):,}')
print(f'censored share: {t.isna().mean():.2%}')

## 2. Sample waterfall

Always print this. The drop from 2.26M to the final sample is the first number a reviewer asks about, and it should be answerable in one table.

In [ ]:
from src.labeling import build_modelling_sample

# Full build is chunked (see the script in the repo history); here on a slice.
sample, audit = build_modelling_sample(
    load_accepted(cfg['paths']['raw_accepted'], nrows=300_000), cfg)
audit

## 3. Performance window sufficiency

Data ends 2018-12. A 36-month loan issued 2015-12 matures exactly at the cutoff, and charge-off is recognised with a lag — hence the buffer. 60-month loans are excluded entirely rather than pooled: one issued in 2015 is still open at the cutoff, which is the same censoring error one level up.

In [ ]:
from src.labeling import check_window_sufficiency

check_window_sufficiency(
    load_accepted(cfg['paths']['raw_accepted'], nrows=300_000), cfg).tail(12)

## 4. Out-of-time splits

There is deliberately no random-split function in `src/splits.py`. A random hold-out trains on 2015 applicants and scores their contemporaries, which measures interpolation. A credit model is asked to hold up on *next* quarter's applicants.

Verified: train 306,462 (13.25% bad) · valid 120,790 (15.12%) · test 162,236 (14.71%).

In [ ]:
from src.splits import split_summary, vintage_table

df = pd.read_parquet('../data/processed/modelling_sample.parquet')
split_summary(df, cfg)

## 5. Vintage curve

First chart to make, last to trust. The bad rate runs 12.33% (2013) to 14.89% (2015) — **the target is not stationary**. That caps how much out-of-time decay can later be blamed on the model: part of any drop is the population moving.

In [ ]:
vintage_table(df, freq='Y')